In [ ]:
import pandas as pd
from paths import ROOT, DATA , INTERIM

In [ ]:
X = pd.read_csv(INTERIM / "v1_features.csv", low_memory=False)
y = pd.read_csv(INTERIM / "v1_labels.csv", low_memory=False)

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    X,
    title="v1 features - univariate profile",
    correlations=None,   # skip bivariate stats (correlation matrices)
    interactions=None,   # skip pairwise scatter interactions
    missing_diagrams=None,
)

profile.to_file(INTERIM / "v1_features_univariate.html")
profile

## Bivariate: each feature vs the label (`default`)

Univariate profiling above looked at each column alone. Here we check each
feature against the label, one at a time - numeric features via correlation,
categorical features via default rate per category.

In [ ]:
data = X.join(y)

numeric_cols = X.select_dtypes(include="number").columns
categorical_cols = X.select_dtypes(exclude="number").columns

# Numeric features: correlation with the 0/1 default label
numeric_vs_label = (
    data[list(numeric_cols) + ["default"]]
    .corr()["default"]
    .drop("default")
    .sort_values(key=abs, ascending=False)
)
print("Numeric features vs default (correlation, strongest first):")
print(numeric_vs_label)

In [ ]:
# Categorical features: default rate per category
for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(
        data.groupby(col)["default"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )